# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohidraheel/Machine-Learning-Practice/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb huggingface_hub pandas numpy scikit-learn

In [2]:
import os
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import login
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

print("Libraries loaded.")

Libraries loaded.


In [3]:
hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN was not found. Add it in Colab Secrets and enable notebook access."
    )

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("SET secret_directory='/tmp'")

con.execute(f'''
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    )
''')

WAREHOUSE_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

con.execute(f'''
    CREATE OR REPLACE VIEW warehouse AS
    SELECT *
    FROM read_parquet(
        '{WAREHOUSE_PATH}',
        hive_partitioning=true
    )
''')

print("Warehouse connected.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Warehouse connected.


In [4]:
schema_df = con.execute("DESCRIBE warehouse").df()
display(schema_df)

required_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_data_available",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_data_available",
    "ga4_sessions",
]

available_columns = schema_df["column_name"].tolist()

missing_columns = [
    column
    for column in required_columns
    if column not in available_columns
]

if missing_columns:
    raise KeyError(f"Missing required warehouse columns: {missing_columns}")

print("Required columns are available.")

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


Required columns are available.


In [6]:
modeling_sql = '''
WITH march AS (
    SELECT
        client_hash_id AS client_id,
        content_hash_id AS content_id,

        SUM(COALESCE(gsc_impressions, 0)) AS impressions_31d,
        SUM(COALESCE(gsc_clicks, 0)) AS clicks_31d,

        CASE
            WHEN SUM(COALESCE(gsc_impressions, 0)) > 0
            THEN SUM(COALESCE(gsc_clicks, 0)) * 1.0
                 / SUM(COALESCE(gsc_impressions, 0))
            ELSE NULL
        END AS ctr_31d,

        CASE
            WHEN SUM(
                CASE
                    WHEN gsc_avg_position IS NOT NULL
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            ) > 0
            THEN SUM(
                CASE
                    WHEN gsc_avg_position IS NOT NULL
                    THEN gsc_avg_position * COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            ) * 1.0
            / SUM(
                CASE
                    WHEN gsc_avg_position IS NOT NULL
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            )
            ELSE NULL
        END AS weighted_position_31d,

        SUM(COALESCE(ga4_sessions, 0)) AS sessions_31d,
        MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS has_ga4_data,
        COUNT(*) AS march_source_rows

    FROM warehouse

    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(COALESCE(gsc_impressions, 0)) >= 100
),

april AS (
    SELECT
        client_hash_id AS client_id,
        content_hash_id AS content_id,
        SUM(COALESCE(gsc_clicks, 0)) AS april_clicks,
        COUNT(*) AS april_source_rows

    FROM warehouse

    WHERE report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30'
      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    m.*,
    a.april_clicks,
    a.april_source_rows,

    CASE
        WHEN m.clicks_31d > 0
        THEN (a.april_clicks - m.clicks_31d) * 1.0 / m.clicks_31d
        ELSE NULL
    END AS future_click_change_pct,

    CASE
        WHEN m.clicks_31d > 0
         AND a.april_clicks <= m.clicks_31d * 0.90
        THEN 1
        ELSE 0
    END AS click_decline_label

FROM march AS m

INNER JOIN april AS a
    USING (client_id, content_id)
'''

model_frame = con.execute(modeling_sql).df()

print("Modeling rows:", len(model_frame))
display(model_frame.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 100893


,client_id,content_id,impressions_31d,clicks_31d,ctr_31d,weighted_position_31d,sessions_31d,has_ga4_data,march_source_rows,april_clicks,april_source_rows,future_click_change_pct,click_decline_label
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,345.0,1.0,0.002899,23.492754,0.0,0,31,0.0,28,-1.0,1
1,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,854.0,1.0,0.001171,8.152225,0.0,0,31,1.0,29,0.0,0
2,client_62f4a7e64f5e0096,content_3f87f49c36774e23,205.0,0.0,0.000000,34.102439,0.0,0,29,0.0,27,NaN,0
3,client_62f4a7e64f5e0096,content_64706c8afebebb8c,666.0,2.0,0.003003,5.243243,0.0,0,31,1.0,30,-0.5,1
4,client_62f4a7e64f5e0096,content_8809e466350d5b98,174.0,0.0,0.000000,6.683908,0.0,0,29,0.0,28,NaN,0
5,client_62f4a7e64f5e0096,content_4b46566297566e02,561.0,0.0,0.000000,8.992870,0.0,0,31,0.0,30,NaN,0
6,client_62f4a7e64f5e0096,content_afec440f43d68a2d,103.0,0.0,0.000000,18.631068,0.0,0,27,0.0,21,NaN,0
7,client_62f4a7e64f5e0096,content_e5ccf652664d63f0,351.0,0.0,0.000000,12.541311,0.0,0,31,0.0,28,NaN,0
8,client_62f4a7e64f5e0096,content_54e631382577b94d,4815.0,6.0,0.001246,0.559086,0.0,0,31,3.0,30,-0.5,1
9,client_62f4a7e64f5e0096,content_87ccab115e180aa9,884.0,0.0,0.000000,29.058824,0.0,0,31,0.0,30,NaN,0


In [7]:
assert len(model_frame) > 0, "The modeling frame is empty."
assert model_frame[["client_id", "content_id"]].notna().all().all()
assert model_frame["impressions_31d"].ge(100).all()
assert model_frame["click_decline_label"].isin([0, 1]).all()

print("Positive-label rate:", round(model_frame["click_decline_label"].mean(), 4))
print("Unique clients:", model_frame["client_id"].nunique())

display(
    model_frame[
        [
            "impressions_31d",
            "clicks_31d",
            "ctr_31d",
            "weighted_position_31d",
            "sessions_31d",
            "april_clicks",
            "future_click_change_pct",
            "click_decline_label",
        ]
    ].describe()
)

Positive-label rate: 0.3934
Unique clients: 43


,impressions_31d,clicks_31d,ctr_31d,weighted_position_31d,sessions_31d,april_clicks,future_click_change_pct,click_decline_label
count,100893.000000,100893.000000,100893.000000,100893.000000,100893.000000,100893.000000,63510.000000,100893.000000
mean,2761.300952,8.078063,0.002620,14.330397,12.092088,7.707462,-0.074485,0.393357
std,6961.642086,34.977339,0.004193,14.742469,44.575190,38.705656,2.162238,0.488497
min,100.000000,0.000000,0.000000,0.015873,0.000000,0.000000,-1.000000,0.000000
25%,286.000000,0.000000,0.000000,4.761118,0.000000,0.000000,-1.000000,0.000000
50%,794.000000,1.000000,0.001248,8.209267,1.000000,1.000000,-0.425000,0.000000
75%,2532.000000,6.000000,0.003676,18.958298,6.000000,4.000000,0.034483,1.000000
max,617124.000000,5668.000000,0.155844,106.890995,2603.000000,7434.000000,289.500000,1.000000


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

March metrics are used as features. April clicks are used only to build the future outcome.

Eligibility requires:

- March GSC data
- At least 100 March impressions
- At least one April GSC row

The label is `1` when April clicks are at least 10% below March clicks. Pages with zero March clicks remain eligible; for them, the label is `0` because a percentage decline cannot be established from a zero baseline.

In [8]:
feature_frame = model_frame.copy()

feature_frame["log_impressions_31d"] = np.log1p(
    feature_frame["impressions_31d"]
)
feature_frame["log_clicks_31d"] = np.log1p(
    feature_frame["clicks_31d"]
)
feature_frame["log_sessions_31d"] = np.log1p(
    feature_frame["sessions_31d"]
)

position_bins = [0, 3, 5, 10, 20, 50, np.inf]
position_labels = ["1-3", "4-5", "6-10", "11-20", "21-50", "51+"]

feature_frame["position_bucket"] = pd.cut(
    feature_frame["weighted_position_31d"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True,
).astype("object")

feature_frame["position_bucket"] = (
    feature_frame["position_bucket"].fillna("Unknown")
)

numeric_features = [
    "log_impressions_31d",
    "log_clicks_31d",
    "ctr_31d",
    "weighted_position_31d",
    "log_sessions_31d",
    "has_ga4_data",
]

categorical_features = [
    "position_bucket",
]

model_features = numeric_features + categorical_features

X = feature_frame[model_features].copy()
y = feature_frame["click_decline_label"].astype(int).copy()
groups = feature_frame["client_id"].copy()

print("X shape:", X.shape)
print("Positive labels:", int(y.sum()))
display(X.head())

X shape: (100893, 7)
Positive labels: 39687


,log_impressions_31d,log_clicks_31d,ctr_31d,weighted_position_31d,log_sessions_31d,has_ga4_data,position_bucket
0,5.846439,0.693147,0.002899,23.492754,0.0,0,21-50
1,6.751101,0.693147,0.001171,8.152225,0.0,0,6-10
2,5.327876,0.000000,0.000000,34.102439,0.0,0,21-50
3,6.502790,1.098612,0.003003,5.243243,0.0,0,6-10
4,5.164786,0.000000,0.000000,6.683908,0.0,0,6-10


In [9]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop=None,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=42,
            ),
        ),
    ]
)

print(model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['log_impressions_31d',
                                                   'log_clicks_31d', 'ctr_31d',
                                                   'weighted_position_31d',
                                                   'log_sessions_31d',
                                                   'has_ga4_data']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a **grouped holdout split by anonymized client**.

This is more honest than a random row split because pages from the same client may share traffic patterns, measurement practices, or content strategy. Keeping clients on only one side reduces the chance that client-specific behavior leaks into evaluation.

The split uses 80% of clients for training and 20% for testing. It is not a full time-series validation because only one feature month and one outcome month are used here.

In [10]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_index, test_index = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_index].copy()
X_test = X.iloc[test_index].copy()

y_train = y.iloc[train_index].copy()
y_test = y.iloc[test_index].copy()

train_meta = feature_frame.iloc[train_index].copy()
test_meta = feature_frame.iloc[test_index].copy()

train_clients = set(train_meta["client_id"])
test_clients = set(test_meta["client_id"])

assert train_clients.isdisjoint(test_clients)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(X_train),
            "clients": len(train_clients),
            "positive_rate": y_train.mean(),
        },
        {
            "split": "test",
            "rows": len(X_test),
            "clients": len(test_clients),
            "positive_rate": y_test.mean(),
        },
    ]
)

display(split_summary)
print("Client overlap:", len(train_clients.intersection(test_clients)))

,split,rows,clients,positive_rate
0,train,85702,34,0.399419
1,test,15191,9,0.359160


Client overlap: 0


In [11]:
if y_train.nunique() < 2:
    raise ValueError(
        "The training split contains only one class. "
        "Change random_state or revisit the label threshold."
    )

if y_test.nunique() < 2:
    print(
        "Warning: the test split contains one class, so ROC-AUC "
        "and average precision may be unavailable."
    )

print("Split checks completed.")

Split checks completed.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The model and baseline use the same test rows and the same binary decline label.

The Week-4-style baseline combines:

- Impression-volume percentile
- Positive CTR gap relative to the median CTR of the page's position bucket

To avoid using the test distribution to define the baseline, expected CTR values and the action threshold are learned from training rows only.

The main comparison metric is **F1-score**, supported by precision and recall.

In [12]:
model.fit(X_train, y_train)

model_probability = model.predict_proba(X_test)[:, 1]
model_prediction = (model_probability >= 0.50).astype(int)

print("Model trained.")

Model trained.


In [13]:
baseline_train = train_meta.copy()
baseline_test = test_meta.copy()

train_expected_ctr = (
    baseline_train
    .groupby("position_bucket", observed=True)["ctr_31d"]
    .median()
    .rename("expected_ctr")
)

global_train_ctr = float(baseline_train["ctr_31d"].median())

for frame in [baseline_train, baseline_test]:
    frame["expected_ctr"] = (
        frame["position_bucket"]
        .map(train_expected_ctr)
        .astype(float)
        .fillna(global_train_ctr)
    )

    frame["ctr_gap"] = (
        frame["expected_ctr"] - frame["ctr_31d"]
    ).clip(lower=0)

def percentile_from_training(train_series, values):
    sorted_values = np.sort(train_series.dropna().to_numpy())

    if len(sorted_values) == 0:
        return np.zeros(len(values), dtype=float)

    return np.searchsorted(
        sorted_values,
        values.to_numpy(),
        side="right",
    ) / len(sorted_values)

baseline_train["volume_percentile"] = (
    baseline_train["impressions_31d"].rank(pct=True)
)
baseline_train["ctr_gap_percentile"] = (
    baseline_train["ctr_gap"].rank(pct=True)
)

baseline_train["baseline_score"] = (
    100 * (
        0.60 * baseline_train["volume_percentile"]
        + 0.40 * baseline_train["ctr_gap_percentile"]
    )
)

baseline_test["volume_percentile"] = percentile_from_training(
    baseline_train["impressions_31d"],
    baseline_test["impressions_31d"],
)

baseline_test["ctr_gap_percentile"] = percentile_from_training(
    baseline_train["ctr_gap"],
    baseline_test["ctr_gap"],
)

baseline_test["baseline_score"] = (
    100 * (
        0.60 * baseline_test["volume_percentile"]
        + 0.40 * baseline_test["ctr_gap_percentile"]
    )
)

positive_rate_train = float(y_train.mean())

if positive_rate_train <= 0:
    baseline_threshold = np.inf
elif positive_rate_train >= 1:
    baseline_threshold = -np.inf
else:
    baseline_threshold = float(
        baseline_train["baseline_score"].quantile(
            1 - positive_rate_train
        )
    )

baseline_prediction = (
    baseline_test["baseline_score"] >= baseline_threshold
).astype(int).to_numpy()

print("Baseline threshold:", round(baseline_threshold, 3))
print("Train positive rate:", round(positive_rate_train, 4))

Baseline threshold: 56.268
Train positive rate: 0.3994


In [14]:
def safe_roc_auc(y_true, probability):
    if pd.Series(y_true).nunique() < 2:
        return np.nan
    return roc_auc_score(y_true, probability)

def safe_average_precision(y_true, probability):
    if pd.Series(y_true).nunique() < 2:
        return np.nan
    return average_precision_score(y_true, probability)

def metric_row(name, y_true, prediction, probability):
    return {
        "method": name,
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "f1": f1_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "roc_auc": safe_roc_auc(y_true, probability),
        "average_precision": safe_average_precision(
            y_true,
            probability,
        ),
        "predicted_positive_rate": np.mean(prediction),
    }

results = pd.DataFrame(
    [
        metric_row(
            "Week-4 baseline",
            y_test,
            baseline_prediction,
            baseline_test["baseline_score"] / 100,
        ),
        metric_row(
            "Logistic regression",
            y_test,
            model_prediction,
            model_probability,
        ),
    ]
)

display(results.round(4))

,method,accuracy,precision,recall,f1,roc_auc,average_precision,predicted_positive_rate
0,Week-4 baseline,0.5541,0.4055,0.5185,0.4551,0.5607,0.4040,0.4592
1,Logistic regression,0.7240,0.5897,0.7612,0.6645,0.8010,0.5906,0.4636


In [15]:
baseline_f1 = float(
    results.loc[
        results["method"] == "Week-4 baseline",
        "f1",
    ].iloc[0]
)

model_f1 = float(
    results.loc[
        results["method"] == "Logistic regression",
        "f1",
    ].iloc[0]
)

f1_difference = model_f1 - baseline_f1

print("Baseline F1:", round(baseline_f1, 4))
print("Model F1:", round(model_f1, 4))
print("Model minus baseline F1:", round(f1_difference, 4))

if f1_difference > 0:
    print("Observed result: the model beat the baseline on this grouped holdout.")
elif f1_difference < 0:
    print("Observed result: the baseline beat the model on this grouped holdout.")
else:
    print("Observed result: the model and baseline tied on F1.")

Baseline F1: 0.4551
Model F1: 0.6645
Model minus baseline F1: 0.2094
Observed result: the model beat the baseline on this grouped holdout.


In [16]:
OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metrics_path = OUTPUT_DIR / "w05_model_metrics.json"

metrics_receipt = {
    "assignment": "ML-08 capstone modeling lane",
    "lane": "CTR opportunity / content opportunity scoring",
    "feature_window": "2026-03-01 to 2026-03-31",
    "outcome_window": "2026-04-01 to 2026-04-30",
    "label": "April clicks at least 10% below March clicks",
    "split": "grouped holdout by client",
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "train_clients": int(len(train_clients)),
    "test_clients": int(len(test_clients)),
    "baseline_f1": baseline_f1,
    "model_f1": model_f1,
    "model_minus_baseline_f1": f1_difference,
    "future_inputs_used_as_features": False,
    "private_fields_used": False,
}

with metrics_path.open("w", encoding="utf-8") as file:
    json.dump(metrics_receipt, file, indent=2)

print("Metrics written to:", metrics_path)
display(pd.Series(metrics_receipt, name="value").to_frame())

Metrics written to: work/outputs/w05_model_metrics.json


,value
assignment,ML-08 capstone modeling lane
lane,CTR opportunity / content opportunity scoring
feature_window,2026-03-01 to 2026-03-31
outcome_window,2026-04-01 to 2026-04-30
label,April clicks at least 10% below March clicks
split,grouped holdout by client
train_rows,85702
test_rows,15191
train_clients,34
test_clients,9


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I inspect false positives and false negatives instead of relying only on one metric.

- **False positive:** The model predicted a decline, but the measured decline label was 0.
- **False negative:** The model predicted no decline, but the measured decline label was 1.

These errors do not establish causality. They show where the chosen March signals did not align with the April outcome.

In [17]:
error_frame = test_meta[
    [
        "client_id",
        "content_id",
        "impressions_31d",
        "clicks_31d",
        "ctr_31d",
        "weighted_position_31d",
        "sessions_31d",
        "april_clicks",
        "future_click_change_pct",
        "click_decline_label",
    ]
].copy()

error_frame["model_probability"] = model_probability
error_frame["model_prediction"] = model_prediction
error_frame["baseline_score"] = baseline_test["baseline_score"].to_numpy()
error_frame["baseline_prediction"] = baseline_prediction

error_frame["error_type"] = np.select(
    [
        (
            (error_frame["model_prediction"] == 1)
            & (error_frame["click_decline_label"] == 0)
        ),
        (
            (error_frame["model_prediction"] == 0)
            & (error_frame["click_decline_label"] == 1)
        ),
    ],
    [
        "false_positive",
        "false_negative",
    ],
    default="correct",
)

print("Model confusion matrix:")
display(
    pd.DataFrame(
        confusion_matrix(y_test, model_prediction),
        index=["actual_0", "actual_1"],
        columns=["predicted_0", "predicted_1"],
    )
)

display(error_frame["error_type"].value_counts().to_frame("n"))

Model confusion matrix:


,predicted_0,predicted_1
actual_0,6845,2890
actual_1,1303,4153


,n
error_type,
correct,10998
false_positive,2890
false_negative,1303


In [18]:
false_positives = (
    error_frame[
        error_frame["error_type"] == "false_positive"
    ]
    .sort_values("model_probability", ascending=False)
    .head(10)
)

false_negatives = (
    error_frame[
        error_frame["error_type"] == "false_negative"
    ]
    .sort_values("model_probability", ascending=True)
    .head(10)
)

print("Highest-confidence false positives")
display(false_positives)

print("Highest-confidence false negatives")
display(false_negatives)

Highest-confidence false positives


,client_id,content_id,impressions_31d,clicks_31d,ctr_31d,weighted_position_31d,sessions_31d,april_clicks,future_click_change_pct,click_decline_label,model_probability,model_prediction,baseline_score,baseline_prediction,error_type
89699,client_2094c6eb080311d5,content_04e3473c74438c83,140.0,11.0,0.078571,2.850000,15.0,20.0,0.818182,0,0.999986,1,29.316935,0,false_positive
86482,client_2094c6eb080311d5,content_055d4079d587e778,111.0,8.0,0.072072,4.882883,9.0,21.0,1.625000,0,0.999961,1,26.200789,0,false_positive
86767,client_2094c6eb080311d5,content_c2c4e02262df304d,307.0,20.0,0.065147,4.710098,25.0,48.0,1.400000,0,0.999930,1,40.605353,0,false_positive
89639,client_2094c6eb080311d5,content_7f3e5c1ec603a7f1,181.0,12.0,0.066298,2.303867,13.0,41.0,2.416667,0,0.999926,1,32.966559,0,false_positive
5696,client_d211cb07b9059bab,content_deb862cd9349169e,157.0,10.0,0.063694,9.146497,5.0,11.0,0.100000,0,0.999897,1,30.920865,0,false_positive
86761,client_2094c6eb080311d5,content_eaa1ca7e37b9b17e,164.0,9.0,0.054878,5.725610,12.0,12.0,0.333333,0,0.999535,1,31.553756,0,false_positive
86478,client_2094c6eb080311d5,content_da8fe29b2620e9b9,453.0,22.0,0.048565,4.030905,24.0,26.0,0.181818,0,0.999303,1,46.151548,0,false_positive
29588,client_0fa64a184f18a4a0,content_9a4459a8a3b7a514,28337.0,803.0,0.028338,3.347037,743.0,2050.0,1.552927,0,0.997995,1,83.946466,1,false_positive
26293,client_3f0ce4d44fe94f3d,content_cbaed40b83f2ec8d,878.0,31.0,0.035308,3.392938,11.0,49.0,0.580645,0,0.997128,1,55.441880,0,false_positive
99282,client_2094c6eb080311d5,content_2fb9a61f91d7245f,145.0,6.0,0.041379,5.510345,6.0,13.0,1.166667,0,0.996419,1,29.793704,0,false_positive


Highest-confidence false negatives


,client_id,content_id,impressions_31d,clicks_31d,ctr_31d,weighted_position_31d,sessions_31d,april_clicks,future_click_change_pct,click_decline_label,model_probability,model_prediction,baseline_score,baseline_prediction,error_type
25800,client_f623b01661d4bfe4,content_f380491a7d7a78f8,444.0,1.0,0.002252,79.245495,0.0,0.0,-1.0,1,0.148449,0,45.886910,0,false_negative
14052,client_f623b01661d4bfe4,content_69ed5aaa4b13da18,412.0,1.0,0.002427,83.342233,4.0,0.0,-1.0,1,0.150488,0,44.834660,0,false_negative
26483,client_3f0ce4d44fe94f3d,content_1c62e6cb4443a446,664.0,1.0,0.001506,71.983434,0.0,0.0,-1.0,1,0.163339,0,51.617932,0,false_negative
1794,client_9958f0a7ae1df715,content_52d6e0ecebc50434,669.0,1.0,0.001495,79.252616,3.0,0.0,-1.0,1,0.166382,0,51.718046,0,false_negative
25816,client_f623b01661d4bfe4,content_cbff21b87e6892da,674.0,1.0,0.001484,77.762611,2.0,0.0,-1.0,1,0.181188,0,51.818861,0,false_negative
50589,client_e547b89c05043229,content_1202038be4d95c1c,1199.0,1.0,0.000834,64.114262,0.0,0.0,-1.0,1,0.190270,0,59.544468,1,false_negative
38813,client_e547b89c05043229,content_3d3537c8be7430cb,848.0,1.0,0.001179,74.745283,2.0,0.0,-1.0,1,0.191020,0,54.975613,0,false_negative
1619,client_9958f0a7ae1df715,content_f2b741f092b30220,366.0,1.0,0.002732,74.095628,2.0,0.0,-1.0,1,0.192973,0,43.153719,0,false_negative
25859,client_3f0ce4d44fe94f3d,content_ea57ddc067408d30,288.0,1.0,0.003472,62.527778,0.0,0.0,-1.0,1,0.194827,0,39.670719,0,false_negative
25821,client_f623b01661d4bfe4,content_fd499895590565e9,251.0,1.0,0.003984,84.633466,1.0,0.0,-1.0,1,0.197535,0,37.750344,0,false_negative


In [19]:
transformed_feature_names = (
    model.named_steps["preprocessor"].get_feature_names_out()
)

coefficients = (
    model.named_steps["classifier"].coef_[0]
)

coefficient_table = pd.DataFrame(
    {
        "feature": transformed_feature_names,
        "coefficient": coefficients,
        "absolute_coefficient": np.abs(coefficients),
    }
).sort_values(
    "absolute_coefficient",
    ascending=False,
)

display(coefficient_table.head(20))

,feature,coefficient,absolute_coefficient
1,numeric__log_clicks_31d,0.613468,0.613468
2,numeric__ctr_31d,0.607784,0.607784
4,numeric__log_sessions_31d,-0.403695,0.403695
0,numeric__log_impressions_31d,0.391614,0.391614
8,categorical__position_bucket_21-50,0.269150,0.269150
5,numeric__has_ga4_data,0.258933,0.258933
3,numeric__weighted_position_31d,-0.231179,0.231179
10,categorical__position_bucket_51+,-0.187732,0.187732
7,categorical__position_bucket_11-20,0.163993,0.163993
6,categorical__position_bucket_1-3,-0.147893,0.147893


### Interpretation

Positive coefficients are directionally associated with a higher predicted probability of the measured click-decline label, while negative coefficients are directionally associated with a lower probability.

Coefficients are not causal effects. Correlated features, client differences, seasonality, and measurement coverage can influence them.

The largest remaining limitations are:

- Only one feature month and one outcome month are used.
- The label measures click decline, not successful refresh impact.
- Search demand and seasonality may change between March and April.
- Query intent and SERP features are not available in this feature set.
- A grouped holdout tests transfer to unseen clients, but repeated time-based validation would be stronger.

In [20]:
privacy_terms = [
    "client_name",
    "company_name",
    "domain",
    "url",
    "query",
    "keyword",
    "page_title",
    "content_text",
]

future_or_label_terms = [
    "april",
    "future",
    "label",
    "target",
    "outcome",
    "next_",
]

privacy_hits = [
    column
    for column in X.columns
    if any(term in column.lower() for term in privacy_terms)
]

leakage_hits = [
    column
    for column in X.columns
    if any(
        term in column.lower()
        for term in future_or_label_terms
    )
]

identifier_hits = [
    column
    for column in ["client_id", "content_id"]
    if column in X.columns
]

print("Privacy hits:", privacy_hits)
print("Future/label hits:", leakage_hits)
print("Identifier hits:", identifier_hits)

assert privacy_hits == []
assert leakage_hits == []
assert identifier_hits == []

print("PASS: No private, future, label-derived, or identifier fields entered X.")

Privacy hits: []
Future/label hits: []
Identifier hits: []
PASS: No private, future, label-derived, or identifier fields entered X.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [21]:
assert train_clients.isdisjoint(test_clients)
assert privacy_hits == []
assert leakage_hits == []
assert identifier_hits == []
assert metrics_path.exists()
assert set(results["method"]) == {
    "Week-4 baseline",
    "Logistic regression",
}

print("ML-08 COMPLETE")
print("Notebook: work/notebooks/w05_model.ipynb")
print("Metrics receipt:", metrics_path)

ML-08 COMPLETE
Notebook: work/notebooks/w05_model.ipynb
Metrics receipt: work/outputs/w05_model_metrics.json
